# Splitting the rent without envy

Three flatmates, three rooms, one rent. Nobody puts a price on a room; each only answers
*at these prices, which room would you take?* Sperner's lemma guarantees prices at which every
flatmate gets a room they picked (Su 1999). Here the flatmates are simulated: each values the
rooms in money and takes the room with the most value for the price.

In [2]:
from sperner import split_rent
from sperner.people import QuasiLinear, envy

rooms = ["balcony", "big", "small"]
people = {
    "Mia": QuasiLinear((1000, 800, 600)),
    "Jonas": QuasiLinear((900, 900, 600)),
    "Lea": QuasiLinear((700, 850, 850)),
}


def ask(person, prices):
    values = [float(prices[room]) for room in rooms]
    return rooms[people[person].choose(values)]


split = split_rent(rooms, 2400, list(people), ask, tolerance=5)
for person, room in split.assignment.items():
    print(f"{person:6} {room:8} {split.prices[room]:8.2f}")
print("questions:", split.questions)

Mia    balcony    998.63
Jonas  big        801.10
Lea    small      600.27
questions: {'Mia': 12, 'Jonas': 6, 'Lea': 6}


## Nobody envies anybody

Envy is how much more a flatmate would gain from another room at its price. It is at most twice
the precision.

In [4]:
prices = [float(split.prices[room]) for room in rooms]
assignment = [rooms.index(split.assignment[p]) for p in people]
print(
    f"largest envy: {envy(list(people.values()), assignment, prices):.2f} "
    f"(precision {float(split.precision):.2f})"
)

largest envy: 0.00 (precision 3.32)


## One question at a time

In an app, answers arrive one by one. `RentSession` asks the next question and can be saved as
JSON in between.

In [6]:
from sperner import RentSession

session = RentSession(rooms, 2400, list(people), tolerance=20)
while (question := session.next_question()) is not None:
    session.answer(ask(question.person, question.prices))
print(session.result.assignment)
print(len(session.to_dict()["answers"]), "answers recorded")

{'Mia': 'balcony', 'Jonas': 'big', 'Lea': 'small'}
20 answers recorded


## Exercises

1. Give Lea a budget of 700 with `Budgeted` from `sperner.people`. What changes?
2. Make the precision ten times finer. How many more questions does it take?
3. With three rooms and only two flatmates, `split_rent` returns prices that work whichever room
   the third takes. Try it and read `plan` in the result.
4. Why does everybody take a free room, and why does that break Sperner's rule 2 unless the
   labels are renamed? (See docs/THEORY.md, section 4.)